# Day 02 Master Lab Walkthrough - Shallow Parsing & Tools for NLP
**Course:** NLP & Computer Vision (PGCP-AI) - CDAC  
**Student:** abs6187 (`23f2000876@ds.study.iitm.ac.in`)  

---

## [Overview] Day 2 Curriculum & University NLP Lab Sequence
This master notebook integrates all **11 core laboratory experiments** taught in Session 3 & Session 4, aligned with the CDAC syllabus and official textbook (*Natural Language Processing* by Prof. Pushpak Bhattacharyya & Aditya Joshi):

| Exp # | Topic | Key Concepts | Tools |
| :--- | :--- | :--- | :--- |
| **Exp 1** | Tokenization of Text Data | Sentence & Word boundaries | `nltk.tokenize` |
| **Exp 2** | Stop Word Removal & Preprocessing | Punctuation filtering, lowercasing | `nltk.corpus.stopwords` |
| **Exp 3** | Part-of-Speech (POS) Tagging | Morphosyntactic word categories | `nltk.pos_tag` |
| **Exp 4** | Stemming using Porter Stemmer | Suffix stripping heuristic rules | `nltk.stem.PorterStemmer` |
| **Exp 5** | Lemmatization using WordNet | Vocabulary-aware morphological base | `nltk.stem.WordNetLemmatizer` |
| **Exp 6** | Morphological Analysis | Prefix, root, and suffix decomposition | `nltk.corpus.wordnet` |
| **Exp 7** | Chunking for Shallow Parsing | Flat syntactic phrases (NP, VP, PP) | `nltk.chunk.RegexpParser` |
| **Exp 8** | Multi-Word Expression (MWE) | Entity preservation (phrases, idioms) | `nltk.tokenize.MWETokenizer` |
| **Exp 9** | Named Entity Recognition (NER) | Information extraction (Persons, Places) | `nltk.ne_chunk`, `spacy` |
| **Exp 10** | Morphological Synthesis | Generating inflected word forms from features | Rule-Based Python Engine |
| **Exp 11** | Word Sense Disambiguation (WSD) | Resolving polysemy via context overlap | NLTK Lesk Algorithm |

> [Note] All experiments are executed on both controlled linguistic sentences and real-world reviews from the Disneyland Reviews dataset.

In [1]:
# [Google Colab & Drive Setup - cdac74abhlak@gmail.com]
# Run this cell if executing on Google Colab to mount Google Drive
import os

try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/NLP_CDAC/notes/day-02'
    os.makedirs(drive_dir, exist_ok=True)
    os.chdir(drive_dir)
    print('[OK] Google Drive linked for cdac74abhlak@gmail.com')
    print('[DIR] Working Directory:', os.getcwd())
    # Uncomment to install dependencies on Colab runtime:
    # !pip install -q nltk spacy pandas
    # !python -m spacy download en_core_web_sm
except ImportError:
    IN_COLAB = False
    print('[OK] Running locally in VS Code / Antigravity IDE (Account: cdac74abhlak@gmail.com)')
    print('[DIR] Working Directory:', os.getcwd())

[OK] Running locally in VS Code / Antigravity IDE (Account: cdac74abhlak@gmail.com)
[DIR] Working Directory: C:\Users\conne\Downloads\NLP_CDAC


---

## Experiment 1: Tokenization of Text Data
**Aim:** Perform sentence and word tokenization on textual data from the Disneyland Reviews dataset.

In [2]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Locate dataset
csv_candidates = ["DisneylandReviews.csv", "notes/day-02/DisneylandReviews.csv", "../../notes/day-02/DisneylandReviews.csv"]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)

if csv_path:
    df = pd.read_csv(csv_path, encoding='latin1', nrows=5)
    review = str(df['Review_Text'][0])
else:
    review = "Disneyland Hong Kong is very fun! We visited Space Mountain and rode the carousel. Great day!"

print("[Original Review Text]:")
print(review[:250] + "...")

# Sentence tokenization
sentences = sent_tokenize(review)
print(f"\n[Sentence Tokenization] ({len(sentences)} sentences):")
for i, s in enumerate(sentences[:3], 1):
    print(f"  {i}. {s}")

# Word tokenization
words = word_tokenize(review)
print(f"\n[Word Tokenization] ({len(words)} total tokens):")
print("  Sample tokens:", words[:15])

[Original Review Text]:
If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides  its a Small World  is absolutely fabulous and worth doing. The day w...

[Sentence Tokenization] (4 sentences):
  1. If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street!
  2. It has a very familiar feel.
  3. One of the rides  its a Small World  is absolutely fabulous and worth doing.

[Word Tokenization] (65 total tokens):
  Sample tokens: ['If', 'you', "'ve", 'ever', 'been', 'to', 'Disneyland', 'anywhere', 'you', "'ll", 'find', 'Disneyland', 'Hong', 'Kong', 'very']


---

## Experiment 2: Stop Word Removal and Text Preprocessing
**Aim:** Implement text preprocessing techniques including punctuation filtering, lowercasing, and stop word removal.

In [3]:
import string
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

sample_text = "Had a wonderful time at Disneyland! The rides were amazing and the staff was extremely friendly."
print("[Original Text]:", sample_text)

# Step 1: Lowercasing
text_lower = sample_text.lower()

# Step 2: Word Tokenization
tokens = word_tokenize(text_lower)

# Step 3: Remove Punctuation
tokens_no_punct = [w for w in tokens if w not in string.punctuation]

# Step 4: Remove English Stop Words
stop_words = set(stopwords.words('english'))
cleaned_tokens = [w for w in tokens_no_punct if w not in stop_words]

print("\n[Tokens without Punctuation]:", tokens_no_punct)
print("[After Stop Word Removal]   :", cleaned_tokens)
print(f"\n[Statistics] Before: {len(tokens)} tokens -> After: {len(cleaned_tokens)} tokens")

[Original Text]: Had a wonderful time at Disneyland! The rides were amazing and the staff was extremely friendly.

[Tokens without Punctuation]: ['had', 'a', 'wonderful', 'time', 'at', 'disneyland', 'the', 'rides', 'were', 'amazing', 'and', 'the', 'staff', 'was', 'extremely', 'friendly']
[After Stop Word Removal]   : ['wonderful', 'time', 'disneyland', 'rides', 'amazing', 'staff', 'extremely', 'friendly']

[Statistics] Before: 18 tokens -> After: 8 tokens


---

## Experiment 3: Part-of-Speech (POS) Tagging
**Aim:** Identify and classify words into their grammatical syntactic categories using NLTK perceptron tagger.

In [4]:
from nltk import pos_tag

nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

pos_sentence = "The excited visitors were riding the Space Mountain roller coaster."
tokens = word_tokenize(pos_sentence)
pos_tags = pos_tag(tokens)

print(f"Sentence: '{pos_sentence}'\n")
print(f"{'Token':<18} | {'POS Tag':<8} | {'Linguistic Category'}")
print("-" * 55)

tag_explanations = {
    "DT": "Determiner",
    "JJ": "Adjective",
    "NNS": "Noun (Plural)",
    "VBD": "Verb (Past Tense)",
    "VBG": "Verb (Present Participle)",
    "NNP": "Proper Noun",
    "NN": "Noun (Singular)",
    ".": "Punctuation"
}

for word, tag in pos_tags:
    desc = tag_explanations.get(tag, tag)
    print(f"{word:<18} | {tag:<8} | {desc}")

Sentence: 'The excited visitors were riding the Space Mountain roller coaster.'

Token              | POS Tag  | Linguistic Category
-------------------------------------------------------
The                | DT       | Determiner
excited            | JJ       | Adjective
visitors           | NNS      | Noun (Plural)
were               | VBD      | Verb (Past Tense)
riding             | VBG      | Verb (Present Participle)
the                | DT       | Determiner
Space              | NNP      | Proper Noun
Mountain           | NNP      | Proper Noun
roller             | NN       | Noun (Singular)
coaster            | NN       | Noun (Singular)
.                  | .        | Punctuation


---

## Experiment 4 & 5: Stemming (Porter) vs. Lemmatization (WordNet)
**Aim:** Compare suffix-stripping heuristic stemming (Porter Stemmer) with vocabulary-aware morphological base extraction (WordNet Lemmatizer).

In [5]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = [
    ("visiting", "v"),
    ("visited", "v"),
    ("rides", "n"),
    ("studies", "v"),
    ("studying", "v"),
    ("feet", "n"),
    ("mice", "n"),
    ("was", "v"),
    ("better", "a")
]

print(f"{'Original Word':<15} | {'POS':<5} | {'Porter Stem':<15} | {'WordNet Lemma'}")
print("-" * 55)

for word, p in test_words:
    stem_val = ps.stem(word)
    lemma_val = lemmatizer.lemmatize(word, pos=p)
    print(f"{word:<15} | {p:<5} | {stem_val:<15} | {lemma_val}")

Original Word   | POS   | Porter Stem     | WordNet Lemma
-------------------------------------------------------


visiting        | v     | visit           | visit
visited         | v     | visit           | visit
rides           | n     | ride            | ride
studies         | v     | studi           | study
studying        | v     | studi           | study
feet            | n     | feet            | foot
mice            | n     | mice            | mouse
was             | v     | wa              | be
better          | a     | better          | good


---

## Experiment 6: Morphological Analysis of Words
**Aim:** Deconstruct words into prefixes, root bases, and suffixes using morphological boundary detection and WordNet validation.

In [6]:
from nltk.corpus import wordnet as wn

prefixes = ["un", "re", "dis", "mis", "pre", "in", "im", "non"]
suffixes = ["ing", "ed", "ly", "ness", "ment", "er", "ful", "less", "es", "s"]

def is_valid_root(r):
    return bool(wn.synsets(r))

def analyze_morphology(w):
    clean_w = w.strip().lower()
    prefix, suffix, root = "None", "None", clean_w
    
    # Prefix detection
    for p in sorted(prefixes, key=len, reverse=True):
        if clean_w.startswith(p):
            cand = clean_w[len(p):]
            if is_valid_root(cand):
                prefix = p
                root = cand
                break
                
    # Suffix detection
    for s in sorted(suffixes, key=len, reverse=True):
        if root.endswith(s):
            cand = root[:-len(s)]
            if is_valid_root(cand):
                suffix = s
                root = cand
                break
                
    return prefix, root, suffix

test_tokens = ["unfriendly", "replaying", "carefully", "disneyland", "disagreement"]

print(f"{'Input Word':<16} | {'Prefix':<10} | {'Root Word':<14} | {'Suffix'}")
print("-" * 55)

for t in test_tokens:
    p, r, s = analyze_morphology(t)
    print(f"{t:<16} | {p:<10} | {r:<14} | {s}")

Input Word       | Prefix     | Root Word      | Suffix
-------------------------------------------------------
unfriendly       | un         | friend         | ly
replaying        | re         | play           | ing
carefully        | None       | careful        | ly
disneyland       | None       | disneyland     | None
disagreement     | dis        | agree          | ment


---

## Experiment 7: Chunking for Shallow Parsing
**Aim:** Implement flat syntactic phrase identification using NLTK `RegexpParser` with grammar `<DT>?<JJ>*<NN.*>+` and chinking.

In [7]:
from nltk.chunk import RegexpParser

chunk_sent = "The magical park has a long waiting line for the exciting roller coaster."
tokens = word_tokenize(chunk_sent)
tagged = pos_tag(tokens)

# Grammar rule: Noun Phrase (NP), Verb Phrase (VP), Prepositional Phrase (PP)
grammar = r"""
    NP: {<DT>?<JJ>*<NN.*>+}
    VP: {<VB.*>+}
    PP: {<IN>}
"""

parser = RegexpParser(grammar)
tree = parser.parse(tagged)

print(f"Sentence: '{chunk_sent}'\n")
print(f"{'Phrase Label':<15} | {'Tokens in Chunk'}")
print("-" * 50)

for subtree in tree.subtrees():
    if subtree.label() in ["NP", "VP", "PP"]:
        phrase_str = " ".join(leaf[0] for leaf in subtree.leaves())
        print(f"{subtree.label():<15} | '{phrase_str}'")

Sentence: 'The magical park has a long waiting line for the exciting roller coaster.'

Phrase Label    | Tokens in Chunk
--------------------------------------------------
NP              | 'The magical park'
VP              | 'has'
NP              | 'a long waiting line'
PP              | 'for'
NP              | 'the exciting roller coaster'


---

## Experiment 8: Multi-Word Expression (MWE) Identification
**Aim:** Preserve compound proper names, phrasal verbs, and idioms using NLTK `MWETokenizer`.

In [8]:
from nltk.tokenize import MWETokenizer

mwe_text = "When you visit Disneyland Paris, you ride the Space Mountain roller coaster and have a blast."
tokens = word_tokenize(mwe_text)

# Registered multi-word tuples
mwe_list = [
    ("Disneyland", "Paris"),
    ("Space", "Mountain"),
    ("roller", "coaster"),
    ("have", "a", "blast")
]

mwe_tok = MWETokenizer(mwe_list, separator='_')
mwe_output = mwe_tok.tokenize(tokens)

print(f"Original Text: '{mwe_text}'\n")
print("[Standard Tokens]     :", tokens[:10])
print("[MWE-Preserved Tokens]:", mwe_output[:8])

print("\nFused Multi-Word Expressions Detected:")
for tok in mwe_output:
    if "_" in tok:
        print(f"  * {tok}")

Original Text: 'When you visit Disneyland Paris, you ride the Space Mountain roller coaster and have a blast.'

[Standard Tokens]     : ['When', 'you', 'visit', 'Disneyland', 'Paris', ',', 'you', 'ride', 'the', 'Space']
[MWE-Preserved Tokens]: ['When', 'you', 'visit', 'Disneyland_Paris', ',', 'you', 'ride', 'the']

Fused Multi-Word Expressions Detected:
  * Disneyland_Paris
  * Space_Mountain
  * roller_coaster
  * have_a_blast


---

## Experiment 9: Named Entity Recognition (NER)
**Aim:** Extract domain entities (Persons, Organizations, Locations, Dates) using NLTK `ne_chunk` and `spaCy`.

In [9]:
import spacy
from nltk import ne_chunk

ner_text = "Walt Disney opened Disneyland in Anaheim, California on July 17, 1955. Over 100 dollars per ticket."

# Method 1: spaCy NER
nlp = spacy.load("en_core_web_sm")
doc = nlp(ner_text)

print(f"Input: '{ner_text}'\n")
print("[spaCy Named Entities]:")
print(f"{'Entity Text':<22} | {'Label':<10} | {'Description'}")
print("-" * 65)

for ent in doc.ents:
    exp = spacy.explain(ent.label_) or ""
    print(f"{ent.text:<22} | {ent.label_:<10} | {exp}")

Input: 'Walt Disney opened Disneyland in Anaheim, California on July 17, 1955. Over 100 dollars per ticket.'

[spaCy Named Entities]:
Entity Text            | Label      | Description
-----------------------------------------------------------------
Walt Disney            | ORG        | Companies, agencies, institutions, etc.
Disneyland             | FAC        | Buildings, airports, highways, bridges, etc.
Anaheim                | GPE        | Countries, cities, states
California             | GPE        | Countries, cities, states
July 17, 1955          | DATE       | Absolute or relative dates or periods
Over 100 dollars       | MONEY      | Monetary values, including unit


---

## Experiment 10: Morphological Synthesis
**Aim:** Develop a rule-based engine to generate inflected word forms from base root words based on target grammatical features.

In [10]:
def synthesize_morphology(root, feature):
    root_l = root.lower()
    
    # Noun Pluralization
    if feature == "plural":
        irregular_plurals = {"child": "children", "mouse": "mice", "foot": "feet"}
        if root_l in irregular_plurals:
            return irregular_plurals[root_l]
        if root_l.endswith(('s', 'x', 'z', 'ch', 'sh')):
            return root + "es"
        if root_l.endswith('y') and root_l[-2] not in "aeiou":
            return root[:-1] + "ies"
        return root + "s"
        
    # Verb Past Tense
    elif feature == "past":
        irregular_past = {"go": "went", "ride": "rode", "see": "saw", "buy": "bought"}
        if root_l in irregular_past:
            return irregular_past[root_l]
        if root_l.endswith('e'):
            return root + "d"
        return root + "ed"
        
    # Verb Continuous Aspect (-ing)
    elif feature == "present_participle":
        if root_l.endswith('e') and not root_l.endswith('ee'):
            return root[:-1] + "ing"
        return root + "ing"
        
    return root

test_roots = ["park", "city", "child", "ride", "go", "visit"]

print(f"{'Root Word':<12} | {'Plural':<14} | {'Past Tense':<14} | {'Present Participle'}")
print("-" * 60)

for r in test_roots:
    pl = synthesize_morphology(r, "plural")
    pa = synthesize_morphology(r, "past")
    pr = synthesize_morphology(r, "present_participle")
    print(f"{r:<12} | {pl:<14} | {pa:<14} | {pr}")

Root Word    | Plural         | Past Tense     | Present Participle
------------------------------------------------------------
park         | parks          | parked         | parking
city         | cities         | cityed         | citying
child        | children       | childed        | childing
ride         | rides          | rode           | riding
go           | gos            | went           | going
visit        | visits         | visited        | visiting


---

## Experiment 11: Word Sense Disambiguation (WSD)
**Aim:** Determine the contextual meaning of polysemous words using the Lesk Algorithm and WordNet.

In [11]:
from nltk.wsd import lesk

s1 = "The fisherman sat on the bank of the river catching fish."
s2 = "She deposited money in the bank and opened an account."
target = "bank"

sense1 = lesk(word_tokenize(s1), target, pos='n')
sense2 = lesk(word_tokenize(s2), target, pos='n')

print(f"Ambiguous Word: '{target}'\n")
print(f"Sentence 1 : '{s1}'")
print(f"Predicted Sense : {sense1.name() if sense1 else 'None'}")
print(f"Definition      : {sense1.definition() if sense1 else 'None'}\n")

print(f"Sentence 2 : '{s2}'")
print(f"Predicted Sense : {sense2.name() if sense2 else 'None'}")
print(f"Definition      : {sense2.definition() if sense2 else 'None'}")

Ambiguous Word: 'bank'

Sentence 1 : 'The fisherman sat on the bank of the river catching fish.'
Predicted Sense : bank.n.01
Definition      : sloping land (especially the slope beside a body of water)

Sentence 2 : 'She deposited money in the bank and opened an account.'
Predicted Sense : depository_financial_institution.n.01
Definition      : a financial institution that accepts deposits and channels the money into lending activities


---

## Summary & Academic Conclusion
This master lab walkthrough successfully demonstrates all 11 foundational NLP components of Session 3 & Session 4:
1. **Low-Level Preprocessing:** Tokenization, stop word filtering, and stemming.
2. **Morphological Foundations:** Analysis, base lemmatization, and rule-based synthesis.
3. **Syntactic Structuring:** POS tagging, shallow parsing with chunking/chinking, and MWE preservation.
4. **Semantic Extraction:** Named Entity Recognition and Word Sense Disambiguation using WordNet.

---
*Created for PGCP-AI CDAC (August 2026 Batch) -- Student abs6187 (`23f2000876@ds.study.iitm.ac.in`)*